# 1. Preparación de Ambiente – Proyecto Venta de Vehículos
Crea la infraestructura en Unity Catalog para la arquitectura **Medallion**:

1. **External Locations** sobre el Data Lake (acceso con el *Storage Credential* `credential`, basado en **Managed Identity**).
2. **Catálogo** `vehiculos` con *managed location* en el container `metastore`.
3. **Esquemas** `raw`, `bronze`, `silver` y `golden`.
4. **Tablas Delta externas** de cada capa.

| Insumo (container `raw`) | Descripción |
|---|---|
| `car_sales.csv` | Transacciones de venta de vehículos (2022-2023) |
| `car_features.csv` | Ficha técnica y precio de lista (MSRP) por marca/modelo/año |
| `dealers.csv` | Catálogo de sucursales (código, ciudad, estado, coordenadas) |

In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
create widget text storageName default "adlsjgl99";

In [0]:
%python
storageName = dbutils.widgets.get("storageName")

### External Locations

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-metastore`
URL 'abfss://metastore@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para el managed location del catálogo';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para los archivos fuente (raw) del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-golden`
URL 'abfss://golden@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas golden del Data Lake';

### Catálogo y esquemas
Se recrea el catálogo y se limpian las rutas físicas para que el proceso sea re-ejecutable. El container `raw` no se toca.

In [0]:
DROP CATALOG IF EXISTS vehiculos CASCADE;

In [0]:
CREATE CATALOG IF NOT EXISTS vehiculos
MANAGED LOCATION 'abfss://metastore@${storageName}.dfs.core.windows.net/'
COMMENT 'Catalogo para la arquitectura medallion del proyecto de venta de vehiculos';

In [0]:
DROP SCHEMA IF EXISTS vehiculos.raw;
DROP SCHEMA IF EXISTS vehiculos.bronze;
DROP SCHEMA IF EXISTS vehiculos.silver;
DROP SCHEMA IF EXISTS vehiculos.golden;

In [0]:
%python
dbutils.fs.rm(f"abfss://bronze@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://silver@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://golden@{storageName}.dfs.core.windows.net/",True)

In [0]:
CREATE SCHEMA IF NOT EXISTS vehiculos.raw;
CREATE SCHEMA IF NOT EXISTS vehiculos.bronze;
CREATE SCHEMA IF NOT EXISTS vehiculos.silver;
CREATE SCHEMA IF NOT EXISTS vehiculos.golden;

### Tablas Bronze

In [0]:
CREATE TABLE IF NOT EXISTS vehiculos.bronze.car_sales (
  car_id          string,
  sale_date       date,
  sale_year       integer,
  customer_name   string,
  gender          string,
  annual_income   integer,
  dealer_name     string,
  company         string,
  model           string,
  engine          string,
  transmission    string,
  color           string,
  price           integer,
  dealer_no       string,
  body_style      string,
  phone           string,
  dealer_region   string,
  source_file     string,
  ingestion_date  timestamp
)
USING DELTA
PARTITIONED BY (sale_year)
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/car_sales"

In [0]:
CREATE TABLE IF NOT EXISTS vehiculos.bronze.car_features (
  make               string,
  model              string,
  year               integer,
  engine_fuel_type   string,
  engine_hp          integer,
  engine_cylinders   integer,
  transmission_type  string,
  driven_wheels      string,
  number_of_doors    integer,
  market_category    string,
  vehicle_size       string,
  vehicle_style      string,
  highway_mpg        integer,
  city_mpg           integer,
  popularity         integer,
  msrp               integer,
  source_file        string,
  ingestion_date     timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/car_features"

In [0]:
CREATE TABLE IF NOT EXISTS vehiculos.bronze.dealers (
  dealer_no       string,
  zip_code        string,
  city            string,
  state_code      string,
  state_name      string,
  latitude        double,
  longitude       double,
  source_file     string,
  ingestion_date  timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/dealers"

### Tablas Silver

In [0]:
CREATE TABLE IF NOT EXISTS vehiculos.silver.car_features_clean (
  make               string,
  model              string,
  latest_year        integer,
  engine_fuel_type   string,
  engine_hp          integer,
  engine_cylinders   integer,
  transmission_type  string,
  driven_wheels      string,
  vehicle_size       string,
  vehicle_style      string,
  highway_mpg        integer,
  city_mpg           integer,
  popularity         integer,
  msrp_avg           double,
  is_luxury          boolean,
  ingestion_date     timestamp
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/car_features_clean"

In [0]:
CREATE TABLE IF NOT EXISTS vehiculos.silver.car_sales_enriched (
  car_id              string,
  sale_date           date,
  sale_year           integer,
  sale_month          integer,
  sale_quarter        integer,
  customer_name       string,
  gender              string,
  annual_income       integer,
  income_segment      string,
  phone               string,
  dealer_name         string,
  dealer_no           string,
  dealer_region       string,
  city                string,
  state_code          string,
  state_name          string,
  latitude            double,
  longitude           double,
  region_consistent   boolean,
  company             string,
  model               string,
  engine              string,
  transmission        string,
  color               string,
  body_style          string,
  price               integer,
  price_segment       string,
  engine_hp           integer,
  vehicle_size        string,
  engine_fuel_type    string,
  msrp_avg            double,
  price_vs_msrp_pct   double,
  has_features        boolean,
  ingestion_date      timestamp
)
USING DELTA
PARTITIONED BY (sale_year)
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/car_sales_enriched"

### Tablas Golden

In [0]:
CREATE TABLE IF NOT EXISTS vehiculos.golden.ventas_mensuales_estado (
  sale_year           integer,
  sale_month          integer,
  state_name          string,
  unidades            long,
  monto_total         long,
  ticket_promedio     double,
  monto_mes_anterior  long,
  variacion_mom_pct   double
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/ventas_mensuales_estado"

In [0]:
CREATE TABLE IF NOT EXISTS vehiculos.golden.ranking_modelos (
  sale_year        integer,
  company          string,
  model            string,
  unidades         long,
  monto_total      long,
  precio_promedio  double,
  ranking_anual    integer
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/ranking_modelos"

In [0]:
CREATE TABLE IF NOT EXISTS vehiculos.golden.kpi_concesionarios (
  dealer_name        string,
  unidades           long,
  monto_total        long,
  ticket_promedio    double,
  participacion_pct  double,
  modelo_top         string
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/kpi_concesionarios"

In [0]:
CREATE TABLE IF NOT EXISTS vehiculos.golden.segmentos_clientes (
  income_segment   string,
  gender           string,
  body_style       string,
  unidades         long,
  monto_total      long,
  ticket_promedio  double
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/segmentos_clientes"